# Causal vs. Predictive Churn Model


## Section 1: Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
import itertools
warnings.filterwarnings('ignore')

CONFIG = {
    "SEED":                  4,
    "N_CUSTOMERS":            500,   # new customers acquired per simulation cycle
    "N_HISTORY":              5000,  # customers in initial historical training batch

    # Causal effect sizes (log-odds scale)
    "SATISFACTION_EFFECT":    -0.6,  # higher satisfaction → less churn
    "TREATMENT_EFFECT":       -1.8,  # treatment reduces churn
    "BASE_CHURN_LOGIT":       0.5,   # baseline churn intercept

    # Historical data: treatment assigned randomly (like an A/B test)
    "HISTORICAL_TREAT_RATE":   0.3,

    # Deployment decision thresholds
    "PREDICTIVE_THRESHOLD":   0.5,   # treat if P(churn) >= this
    "UPLIFT_THRESHOLD":       -0.05, # treat if estimated uplift <= this

    # Simulation controls
    "SIM_CYCLES":             52,
    "EPSILON":                0.05,  # random exploration rate (prevents data collapse)
    "WINDOW_CYCLES":          10,

    # Treatment quarantine window (cycles after treatment where customer cannot be treated again)
    "TREATMENT_COOLDOWN":      4,

    # --- Business parameters (edit to your scenario) ---
    "COST_TREATMENT":         5.0,   # cost per treated customer
    "COST_CHURN":             50.0,  # cost if a customer churns (lost margin / winback cost)
    "REVENUE_IF_RETAINED":     20.0,  # revenue if customer stays this cycle
}

np.random.seed(CONFIG["SEED"])

# Persistent customer ID generator (guarantees no overlaps across cycles)
id_gen = itertools.count(start=1)


## Data Generating Process

In [ ]:
# Reused from supervisor's notebook (simplified)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def generate_customers(n, id_gen, treat_rate=None, treatment_array=None):
    satisfaction = np.clip(np.random.normal(5, 2, n), 0, 10)

    if treatment_array is not None:
        treatment = treatment_array.astype(int)
    elif treat_rate is not None:
        treatment = np.random.binomial(1, treat_rate, n)
    else:
        treatment = np.zeros(n, dtype=int)

    logit_churn = (
        CONFIG["BASE_CHURN_LOGIT"]
        + CONFIG["SATISFACTION_EFFECT"] * satisfaction
        + CONFIG["TREATMENT_EFFECT"] * treatment
    )
    p_churn = sigmoid(logit_churn)
    churn = np.random.binomial(1, p_churn)

    # True treatment effect per customer (counterfactual — unobservable in practice)
    p_if_treated   = sigmoid(CONFIG["BASE_CHURN_LOGIT"] + CONFIG["SATISFACTION_EFFECT"] * satisfaction + CONFIG["TREATMENT_EFFECT"])
    p_if_untreated = sigmoid(CONFIG["BASE_CHURN_LOGIT"] + CONFIG["SATISFACTION_EFFECT"] * satisfaction)
    tau_true = p_if_treated - p_if_untreated  # negative = treatment reduces churn

    return pd.DataFrame({
        "customer_id": [next(id_gen) for _ in range(n)],
        "satisfaction": satisfaction,
        "treatment":    treatment,
        "churn":        churn,
        "p_churn":      p_churn,
        "tau_true":     tau_true
    })


# Sanity check
test_df = generate_customers(10000, id_gen, treat_rate=CONFIG["HISTORICAL_TREAT_RATE"])
print(f"Overall churn rate:     {test_df.churn.mean():.1%}")
print(f"Untreated churn rate:   {test_df[test_df.treatment==0].churn.mean():.1%}")
print(f"Treated churn rate:     {test_df[test_df.treatment==1].churn.mean():.1%}")
print(f"Avg tau_true:           {test_df.tau_true.mean():.3f}")

## Section 3: Initial Training Data

In [ ]:
initial_data = generate_customers(
    CONFIG["N_HISTORY"],
    id_gen,
    treat_rate=CONFIG["HISTORICAL_TREAT_RATE"]
)

initial_data['cycle'] = 0


# Initialize policy-specific active customer bases at cycle 0
base_cols = ["customer_id", "satisfaction", "tau_true"]
active_pred = initial_data[base_cols].copy()
active_causal = initial_data[base_cols].copy()
for df in (active_pred, active_causal):
    df["join_cycle"] = 0
    df["last_treated_cycle"] = -10**9  # effectively 'never treated'

print(f"Training set size:      {len(initial_data):,}")
print(f"Churn rate:             {initial_data.churn.mean():.1%}")
print(f"Treatment rate:         {initial_data.treatment.mean():.1%}")

In [ ]:
initial_data


## Training the models

In [ ]:
# training the models
def train_predictive_model(df):
    X = df[['satisfaction']]
    y = df['churn']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=CONFIG['SEED'])

    predictive_model = LogisticRegression()

    predictive_model.fit(X_train, y = y_train)

    f1 = f1_score(y_test, predictive_model.predict(X_test))

    return {'model' : predictive_model, 'f1_score' : f1}

def train_causal_model(df):
    df_1 = df[df['treatment'] == 1]
    df_0 = df[df['treatment'] == 0]

    X_1 = df_1[['satisfaction']]
    y_1 = df_1['churn']

    X_0 = df_0[['satisfaction']]
    y_0 = df_0['churn']

    X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_1, y_1, test_size = 0.25, random_state=CONFIG['SEED'])
    X_train_0, X_test_0, y_train_0, y_test_0 = train_test_split(X_0, y_0, test_size = 0.25, random_state=CONFIG['SEED'])

    m1 = LogisticRegression()
    m0 = LogisticRegression()

    m1.fit(X_train_1, y_train_1)
    m0.fit(X_train_0, y_train_0)

    # Calculating predictions
    preds = np.concatenate([m1.predict(X_test_1),
                           m0.predict(X_test_0)])
    
    y_true = np.concatenate([y_test_1, y_test_0])

    f1 = f1_score(y_true, preds)

    return {'m1' : m1, 'm0' : m0, 'f1_score' : f1}

def predict_uplift(m1, m0, X):
    p1 = m1.predict_proba(X)[:,1]
    p0 = m0.predict_proba(X)[:,1]
    
    return p1 - p0


## The deployment loop 

In [ ]:
metrics = []

# Copying initial dataset to keep separate across policies
initial_data_pred = initial_data.copy()
initial_data_causal = initial_data.copy()

# Training the models on the initial data
predicitve_model = train_predictive_model(initial_data_pred)
causal_model = train_causal_model(initial_data_causal)

for cycle in range(1, CONFIG["SIM_CYCLES"] + 1):
    # --- 1) Acquire new customers and add to each policy's active base ---
    new_customers = generate_customers(CONFIG['N_CUSTOMERS'], id_gen)
    new_customers['join_cycle'] = cycle
    new_customers['last_treated_cycle'] = -10**9

    active_pred = pd.concat([
        active_pred,
        new_customers[["customer_id", "satisfaction", "tau_true", "join_cycle", "last_treated_cycle"]]
    ], ignore_index=True)

    active_causal = pd.concat([
        active_causal,
        new_customers[["customer_id", "satisfaction", "tau_true", "join_cycle", "last_treated_cycle"]]
    ], ignore_index=True)

    # --- 2) Decide treatments on the ACTIVE base (respecting cooldown) ---
    eligible_pred = (cycle - active_pred["last_treated_cycle"]) > CONFIG["TREATMENT_COOLDOWN"]
    eligible_causal = (cycle - active_causal["last_treated_cycle"]) > CONFIG["TREATMENT_COOLDOWN"]

    # Predictive policy
    X_score_pred = active_pred[["satisfaction"]]
    pred_churn_prob = predicitve_model['model'].predict_proba(X_score_pred)[:, 1]
    pred_treat = (pred_churn_prob >= CONFIG['PREDICTIVE_THRESHOLD']).astype(int)

    # Epsilon-greedy exploration, but only among eligible customers
    explore_mask = (np.random.rand(len(active_pred)) < CONFIG["EPSILON"]).astype(int)
    pred_treat = np.maximum(pred_treat, explore_mask)
    pred_treat = (pred_treat * eligible_pred.astype(int)).astype(int)

    # Causal (uplift) policy
    X_score_causal = active_causal[["satisfaction"]]
    estimated_uplift = predict_uplift(causal_model['m1'], causal_model['m0'], X_score_causal)
    causal_treat = (estimated_uplift <= CONFIG['UPLIFT_THRESHOLD']).astype(int)

    explore_mask_c = (np.random.rand(len(active_causal)) < CONFIG["EPSILON"]).astype(int)
    causal_treat = np.maximum(causal_treat, explore_mask_c)
    causal_treat = (causal_treat * eligible_causal.astype(int)).astype(int)

    # --- 3) Simulate outcomes under each policy ---
    pred_observed = active_pred.copy()
    causal_observed = active_causal.copy()

    pred_observed['treatment'] = pred_treat
    causal_observed['treatment'] = causal_treat

    pred_observed = apply_treatment(pred_observed)
    causal_observed = apply_treatment(causal_observed)

    # Update last treated cycle for those treated this cycle
    pred_observed.loc[pred_observed['treatment'] == 1, 'last_treated_cycle'] = cycle
    causal_observed.loc[causal_observed['treatment'] == 1, 'last_treated_cycle'] = cycle

    # Add cycle stamp and business columns
    pred_observed['cycle'] = cycle
    causal_observed['cycle'] = cycle

    pred_observed = add_business(pred_observed)
    causal_observed = add_business(causal_observed)

    # --- 4) Update ACTIVE bases (keep only non-churned customers) ---
    active_pred = pred_observed[pred_observed['churn'] == 0][
        ["customer_id", "satisfaction", "tau_true", "join_cycle", "last_treated_cycle"]
    ].copy()

    active_causal = causal_observed[causal_observed['churn'] == 0][
        ["customer_id", "satisfaction", "tau_true", "join_cycle", "last_treated_cycle"]
    ].copy()

    # --- 5) Add to historical training data (windowed) ---
    initial_data_pred = pd.concat([initial_data_pred, pred_observed], ignore_index=True)
    initial_data_causal = pd.concat([initial_data_causal, causal_observed], ignore_index=True)

    initial_data_pred = initial_data_pred[initial_data_pred["cycle"] > cycle - CONFIG["WINDOW_CYCLES"]]
    initial_data_causal = initial_data_causal[initial_data_causal["cycle"] > cycle - CONFIG["WINDOW_CYCLES"]]

    # --- 6) Re-train models on latest window ---
    predicitve_model = train_predictive_model(initial_data_pred)
    causal_model = train_causal_model(initial_data_causal)

    # --- 7) Evaluation metrics ---
    true_persuadables_pred = (pred_observed["tau_true"] <= CONFIG["UPLIFT_THRESHOLD"]).values
    true_persuadables_causal = (causal_observed["tau_true"] <= CONFIG["UPLIFT_THRESHOLD"]).values

    def policy_f1(treat_decisions, true_persuadables):
        return f1_score(true_persuadables, treat_decisions, zero_division=0)

    metrics.append({
        "cycle": cycle,

        # Active customer base (end of cycle, after churn)
        "pred_active_base": len(active_pred),
        "causal_active_base": len(active_causal),

        # Treatment rates (within the acted-on base)
        "pred_treat_rate": pred_observed["treatment"].mean(),
        "causal_treat_rate": causal_observed["treatment"].mean(),

        # Churn rates under each policy
        "pred_churn_rate": pred_observed["churn"].mean(),
        "causal_churn_rate": causal_observed["churn"].mean(),

        # Holdout ML metric (misleading for causal!)
        "pred_f1_test": predicitve_model["f1_score"],
        "causal_f1_test": causal_model["f1_score"],

        # --- Business metrics (per-cycle) ---
        "pred_treatment_cost": pred_observed["treatment_cost"].sum(),
        "causal_treatment_cost": causal_observed["treatment_cost"].sum(),
        "pred_churn_cost": pred_observed["churn_cost"].sum(),
        "causal_churn_cost": causal_observed["churn_cost"].sum(),
        "pred_total_cost": pred_observed["total_cost"].sum(),
        "causal_total_cost": causal_observed["total_cost"].sum(),
        "pred_net_value": pred_observed["net_value"].sum(),
        "causal_net_value": causal_observed["net_value"].sum(),

        # Targeting quality (vs. ground truth persuadables)
        "pred_policy_f1": policy_f1(pred_observed["treatment"].values, true_persuadables_pred),
        "causal_policy_f1": policy_f1(causal_observed["treatment"].values, true_persuadables_causal),

        # Training data composition
        "pred_train_treat_rate": initial_data_pred["treatment"].mean(),
        "causal_train_treat_rate": initial_data_causal["treatment"].mean(),
        "pred_train_churn_rate": initial_data_pred["churn"].mean(),
        "causal_train_churn_rate": initial_data_causal["churn"].mean(),
    })

    print(
        f" Cycle {cycle:2d}/{CONFIG['SIM_CYCLES']}"
        f" | Active base (Pred/Causal): {len(active_pred)}/{len(active_causal)}"
        f" | Churn (Pred/Causal): {pred_observed['churn'].mean():.1%}/{causal_observed['churn'].mean():.1%}"
        f" | Net value (Pred/Causal): {metrics[-1]['pred_net_value']:.0f}/{metrics[-1]['causal_net_value']:.0f}"
    )

df_metrics = pd.DataFrame(metrics)
print("\nSimulation complete.")


## Visaulizing the results

In [ ]:
df_metrics = pd.DataFrame(metrics)

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Causal vs. Predictive Model", fontsize=15, fontweight="bold")

cycles = df_metrics["cycle"]
RED, BLUE = "#e74c3c", "#2980b9"

# Plot 1: Policy F1
ax = axs[0, 0]
ax.plot(cycles, df_metrics["pred_policy_f1"],   color=RED,  marker="o", label="Predictive")
ax.plot(cycles, df_metrics["causal_policy_f1"], color=BLUE, marker="s", label="T-Learner (Causal)")
ax.set_title("Policy F1\n(vs. ground truth persuadables)", fontsize=11)
ax.set_ylabel("F1 Score")
ax.set_xlabel("Deployment Cycle")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# Plot 2: Churn rate
ax = axs[0, 1]
ax.plot(cycles, df_metrics["pred_churn_rate"],   color=RED,  marker="o", label="Predictive")
ax.plot(cycles, df_metrics["causal_churn_rate"], color=BLUE, marker="s", label="T-Learner (Causal)")
ax.set_title("Observed Churn Rate", fontsize=11)
ax.set_ylabel("Churn Rate")
ax.set_xlabel("Deployment Cycle")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Holdout Test F1
ax = axs[1, 0]
ax.plot(cycles, df_metrics["pred_f1_test"],   color=RED,  marker="o", label="Predictive")
ax.plot(cycles, df_metrics["causal_f1_test"], color=BLUE, marker="s", label="T-Learner (Causal)")
ax.set_title("Holdout Test F1: The 'Accuracy Trap'\n", fontsize=11)
ax.set_ylabel("Test F1 Score")
ax.set_xlabel("Deployment Cycle")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 0.2)

# Plot 4: Active customer base
ax = axs[1, 1]
ax.plot(cycles, df_metrics['pred_active_base'], color=RED, marker='o', label='Predictive')
ax.plot(cycles, df_metrics['causal_active_base'], color=BLUE, marker='s', label='T-Learner (Causal)')
ax.set_title('Active Customer Base (end of cycle)', fontsize=11)
ax.set_ylabel('Active customers')
ax.set_xlabel('Deployment Cycle')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Additional diagnostic: training window treatment-rate drift ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cycles, df_metrics['pred_train_treat_rate'], color=RED, marker='o', label='Predictive')
ax.plot(cycles, df_metrics['causal_train_treat_rate'], color=BLUE, marker='s', label='T-Learner (Causal)')
ax.axhline(CONFIG['HISTORICAL_TREAT_RATE'], color='gray', linestyle='--', alpha=0.6,
           label=f"Initial random rate ({CONFIG['HISTORICAL_TREAT_RATE']:.0%})")
ax.set_title('Training Window: Treatment Rate Over Time', fontsize=11)
ax.set_ylabel('Fraction treated')
ax.set_xlabel('Deployment Cycle')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

